In [ ]:
pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 32.7 MB/s eta 0:00:00


In [ ]:
!pip uninstall nltk -y
!pip install nltk


Found existing installation: nltk 3.9.1
Uninstalling nltk-3.9.1:
  Successfully uninstalled nltk-3.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.1 MB/s eta 0:00:00


In [ ]:
!pip install gensim

In [ ]:
from IPython.display import FileLink
import pandas as pd
import numpy as np
import gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from nltk.tokenize import word_tokenize
import re
import joblib
import nltk
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer
from nltk.stem.snowball import RussianStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pymorphy3
from collections import Counter
from tqdm.auto import tqdm
from gensim.models import FastText
tqdm.pandas()
from tqdm import tqdm
import string
from gensim.models import FastText
from nltk.stem import WordNetLemmatizer

In [ ]:
df1 = pd.read_csv('negative.csv', delimiter=';', names=['id', 'date', 'name', 'text', 'typr', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])
df2 = pd.read_csv('positive.csv', delimiter=';', names=['id', 'date', 'name', 'text', 'typr', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])
df = pd.concat([df1, df2])
df = df.drop(['id', 'date', 'name', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'], axis=1)
df.to_csv('combined_data.csv', index=False)

In [ ]:
df

,text,typr
0,на работе был полный пиддес :| и так каждое за...,-1
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1
2,@elina_4post как говорят обещаного три года жд...,-1
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1
...,...,...
114906,"Спала в родительском доме, на своей кровати......",1
114907,RT @jebesilofyt: Эх... Мы немного решили сокра...,1
114908,"Что происходит со мной, когда в эфире #proacti...",1
114909,"""Любимая,я подарю тебе эту звезду..."" Имя како...",1


In [ ]:
nltk.download('stopwords')
morph = MorphAnalyzer()
stemmer = RussianStemmer(True)

default_stopwords = set(stopwords.words('russian'))
custom_exceptions = {'нет', 'не', 'опять', 'разве', 'хорошо',
                     'ни', 'ничего',  'наконец',
                     'совсем', 'уже', 'очень', 'чересчур',
                     'слишком', 'никогда'}

filtered_stopwords = {word for word in default_stopwords if word not in custom_exceptions}

# Функция нормализации
def preprocess_text(text, stop_words=filtered_stopwords):
    text = text.lower()
    text = re.sub(r'[^а-яё0-9 ]', '', text)  # Оставляем только буквы, цифры и пробелы
    text = re.sub(r'\s+', ' ', text).strip()  # Убираем лишние пробелы
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words]
    lemmatized = [morph.parse(token)[0].normal_form for token in tokens]
    return ' '.join(lemmatized)

df['text'] = df['text'].progress_apply(lambda x: preprocess_text(str(x)))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


  0%|          | 0/226834 [00:00<?, ?it/s]

In [ ]:
df.head()

,text,typr
0,работа полный пиддес каждый закрытие месяц сви...,-1
1,коллега сидеть рубиться изз долбать винд не мочь,-1
2,4 говорить обещаной год ждать,-1
3,желать хороший полёт удачный посадкия быть оче...,-1
4,обновить какимтый леший не работать простоплеер,-1


LinisCrowd 2015

In [ ]:
df_dict = pd.read_csv('words_all_full_rating.csv', delimiter=';', encoding='windows-1251')
df_dict = df_dict[['Words', 'mean']].dropna()
df_dict['Words'] = df_dict['Words'].apply(lambda x: x.lower())
df_dict['mean'] = df_dict['mean'].str.replace(',', '.').astype(float)

tone_dict = dict(zip(df_dict['Words'], df_dict['mean']))

def extract_tone_features(text, tone_dict=tone_dict):
    tokens = text.split()
    tone_scores = [tone_dict[word] for word in tokens if word in tone_dict]

    if not tone_scores:
        tone_scores = [0]

    tone_scores = np.array(tone_scores, dtype=float)

    mean = np.mean(tone_scores)
    max_value = np.max(tone_scores)
    min_value = np.min(tone_scores)
    total = np.sum(tone_scores)
    pos_count = len([x for x in tone_scores if x > 0])
    neg_count = len([x for x in tone_scores if x < 0])

    return [mean, max_value, min_value, total, pos_count, neg_count]

tqdm.pandas()
df[['mean', 'max', 'min', 'sum', 'pos_count', 'neg_count']] = df['text'].progress_apply(
    lambda x: pd.Series(extract_tone_features(x))
)
df.to_csv('linguistic_features.csv', index=False)

100%|██████████| 226834/226834 [00:36<00:00, 6186.68it/s] 


PoS

In [ ]:
pos_tags = ['NOUN', 'VERB', 'ADJF', 'ADJS', 'ADVB', 'NUMR', 'PRTF',
            'PRTS', 'NPRO', 'CONJ', 'PREP', 'INTJ', 'COMP', 'INFN']

morph = pymorphy3.MorphAnalyzer()

def extract_morph_features(text):
    counter = Counter()
    tokens = text.split()
    total_tokens = len(tokens)

    if total_tokens == 0:
        return [0] * len(pos_tags)

    for token in tokens:
        parsed = morph.parse(token)[0]
        if parsed.tag.POS in pos_tags:
            counter[parsed.tag.POS] += 1

    return [counter[pos] / total_tokens if total_tokens > 0 else 0 for pos in pos_tags]

df_morph = pd.DataFrame()

tqdm.pandas()
df_morph[pos_tags] = df['text'].progress_apply(lambda x: pd.Series(extract_morph_features(x)))

df_morph.to_csv('morphological_features.csv', index=False)


100%|██████████| 226834/226834 [04:21<00:00, 867.26it/s] 


TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words=list(filtered_stopwords), lowercase=True)


In [ ]:
tfidf_matrix = vectorizer.fit_transform(df['text'])
print("Размер матрицы TF-IDF:", tfidf_matrix.shape)


Размер матрицы TF-IDF: (226834, 1000)


In [ ]:
# Сохранение в файл
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
df_tfidf.to_csv('tfidf_features.csv', index=False)

In [ ]:
df_tfidf

,00,01,09,10,100,11,111213,12,13,14,...,штука,шутка,экзамен,эмоция,это,эх,юм,язык,январь,ёлка
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
226830,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.329668,0.0,0.0,0.0,0.0
226831,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
226832,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


#STOP

Word2vec

In [1]:
df['tokens'] = df['text'].apply(lambda x: x.split())

# Обучаем модель Word2Vec на нашем корпусе
w2v_model = Word2Vec(sentences=df['tokens'].tolist(), vector_size=300, window=5, min_count=1, workers=4)

# Функция для усреднения векторов слов
def get_avg_w2v(tokens, model):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Извлекаем признаки для каждого сообщения
df['w2v_features'] = df['tokens'].apply(lambda tokens: get_avg_w2v(tokens, w2v_model))
w2v_features = np.vstack(df['w2v_features'].values)
df_w2v = pd.DataFrame(w2v_features)
df_w2v.to_csv('w2v_features.csv', index=False)

NameError: name 'df' is not defined

In [ ]:
df_w2v

Предобученные Word2vec-признаки

In [ ]:
pretrained_w2v = KeyedVectors.load_word2vec_format('model.bin', binary=True, encoding='utf-8', unicode_errors='ignore')

def get_avg_pretrained_w2v(tokens, model):
    vectors = [model[token] for token in tokens if token in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df['w2v_pretrained_features'] = df['tokens'].apply(lambda tokens: get_avg_pretrained_w2v(tokens, pretrained_w2v))
w2v_pretrained_features = np.vstack(df['w2v_pretrained_features'].values)
df_w2v_pretrained = pd.DataFrame(w2v_pretrained_features)
df_w2v_pretrained.to_csv('w2v_pretrained_features.csv', index=False)

In [ ]:
df_w2v_pretrained

Fasttext-признаки, обученные самостоятельно.

In [ ]:
#pip install gensim nltk pymorphy2 tqdm


In [ ]:
'''fasttext_model = FastText(sentences=df['tokens'].tolist(), vector_size=300, window=5, min_count=1, workers=4)

def get_avg_fasttext(tokens, model):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df['fasttext_features'] = df['tokens'].apply(lambda tokens: get_avg_fasttext(tokens, fasttext_model))
fasttext_features = np.vstack(df['fasttext_features'].values)
df_fasttext = pd.DataFrame(fasttext_features)
df_fasttext.to_csv('fasttext_features.csv', index=False)'''

In [ ]:
df_fasttex

Предобученные Fasttext -признаки.

In [ ]:
'''pretrained_fasttext = KeyedVectors.load('fasttext_pretrained.model')  # Замените имя файла, если требуется

def get_avg_pretrained_fasttext(tokens, model):
    vectors = [model[token] for token in tokens if token in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df['fasttext_pretrained_features'] = df['tokens'].apply(lambda tokens: get_avg_pretrained_fasttext(tokens, pretrained_fasttext))
fasttext_pretrained_features = np.vstack(df['fasttext_pretrained_features'].values)
df_fasttext_pretrained = pd.DataFrame(fasttext_pretrained_features)
df_fasttext_pretrained.to_csv('fasttext_pretrained_features.csv', index=False)'''


In [ ]:
'''import gc
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

results_lr = {}
results_svc = {}

def get_vectors(path):
    data = pd.read_csv(path, low_memory=True)
    features = data.drop('typr', axis=1)
    labels = data['typr']
    return features, labels

def train(features, labels, model, data_type):
    X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if model.__class__.__name__ == 'LogisticRegression':
        results_lr[data_type] = accuracy_score(y_test, y_pred)
    elif model.__class__.__name__ == 'SVC':
        results_svc[data_type] = accuracy_score(y_test, y_pred)
    print(f"Результаты для {data_type}:")
    print(classification_report(y_test, y_pred))
    print("="*50)

# Инициализация моделей
lr_model = LogisticRegression(max_iter=1000)
svc_model = SVC()

# Список путей и типов признаков
feature_files = [
    (r'linguistic_features.csv', 'linis crowd'),
    (r'morphological_features.csv', 'speech parts'),
    (r'tfidf.csv', 'tfidf'),
    (r'w2v.csv', 'wav2vec'),
    (r'w2v_pretrained.csv', 'wav2vec_pretrained'),
    (r'fasttext.csv', 'fast text'),
    #(r'fasttext_pretr.csv', 'fast text pretrained')
]

for path, dtype in feature_files:
    features, labels = get_vectors(path)
    # Обучаем и оцениваем для LogisticRegression
    train(features, labels, lr_model, dtype)
    # Обучаем и оцениваем для SVC
    train(features, labels, svc_model, dtype)
    # После обработки текущего набора освобождаем память
    del features, labels
    gc.collect()

# Вывод результатов
results_lr_df = pd.DataFrame.from_dict(results_lr, orient='index', columns=['Accuracy'])
results_svc_df = pd.DataFrame.from_dict(results_svc, orient='index', columns=['Accuracy'])

print("Logistic Regression результаты:")
print(results_lr_df)
print("\nSVC результаты:")
print(results_svc_df)'''


In [ ]:
#df_w2v = pd.read_csv('w2v_features.csv')


In [ ]:
#df_w2v_pretrained = pd.read_csv('w2v_pretrained_features.csv')


In [ ]:
#df_fasttext = pd.read_csv('fasttext_features.csv')
#df_fasttext_pretrained = pd.read_csv('fasttext_pretrained_features.csv')



In [ ]:
#lr_model = LogisticRegression(max_iter=1000)
#svc_model = SVC()

In [ ]:
#train_and_evaluate(df_tfidf, labels, lr_model, "Logistic Regression", "TF-IDF")
#train_and_evaluate(df_tfidf, labels, svc_model, "SVC", "TF-IDF")

In [ ]:
#train_and_evaluate(df_w2v, labels, lr_model, "Logistic Regression", "Word2Vec (обученная)")
#train_and_evaluate(df_w2v, labels, svc_model, "SVC", "Word2Vec (обученная)")

In [ ]:
#train_and_evaluate(df_w2v_pretrained, labels, lr_model, "Logistic Regression", "Word2Vec (предобученная)")
#train_and_evaluate(df_w2v_pretrained, labels, svc_model, "SVC", "Word2Vec (предобученная)")

In [ ]:
#train_and_evaluate(df_fasttext, labels, lr_model, "Logistic Regression", "FastText (обученная)")
#train_and_evaluate(df_fasttext, labels, svc_model, "SVC", "FastText (обученная)")

In [2]:
#train_and_evaluate(df_fasttext_pretrained, labels, lr_model, "Logistic Regression", "FastText (предобученная)")
#train_and_evaluate(df_fasttext_pretrained, labels, svc_model, "SVC", "FastText (предобученная)")

In [ ]:
##############################################
# Дополнение: пункты 8-15 согласно первоначальному плану
##############################################

# ----------------------------
# (8) Обучение модели Word2Vec и извлечение признаков
# ----------------------------
# (Уже реализовано: обучаем модель на токенах и вычисляем средний вектор)
# Результат сохранён в файле 'w2v_features.csv'
# Теперь объединяем эти признаки с основным DataFrame и переименовываем столбец метки:
df.rename(columns={'typr': 'type'}, inplace=True)
w2v_features_df = pd.DataFrame(w2v_features, columns=[f'w2v_{i}' for i in range(w2v_features.shape[1])])
df = pd.concat([df.reset_index(drop=True), w2v_features_df.reset_index(drop=True)], axis=1)

# ----------------------------



In [ ]:
# (9) Классификация (Logistic Regression) с использованием Word2Vec-признаков
# ----------------------------
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

X = df[[f'w2v_{i}' for i in range(w2v_features.shape[1])]]
y = df['type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

print("Logistic Regression (Word2Vec features):")
print(classification_report(y_test, y_pred))
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

# ----------------------------


In [ ]:
pip install ufal_udpipe

In [ ]:
# (10) Предобработка текста с использованием UDPipe
# ----------------------------
from ufal.udpipe import Model, Pipeline

ud_model_path = "udpipe_syntagrus.model"  # Укажите корректный путь к модели UDPipe
ud_model = Model.load(ud_model_path)
if not ud_model:
    raise Exception("Не удалось загрузить модель UDPipe.")
ud_pipeline = Pipeline(ud_model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

def preprocess_text_with_udpipe(text):
    processed = ud_pipeline.process(text)
    lines = processed.split("\n")
    lemmas = []
    for line in lines:
        if line.startswith("#") or line.strip() == "":
            continue
        parts = line.split("\t")
        if len(parts) != 10:
            continue
        lemma = parts[2]  # лемма
        pos = parts[3]    # часть речи
        lemmas.append(f"{lemma}_{pos}")
    return " ".join(lemmas)

# Создаём копию DataFrame для UDPipe обработки
df_udpipe = df.copy()
df_udpipe['processed_text'] = df_udpipe['text'].apply(preprocess_text_with_udpipe)
df_udpipe['tokens'] = df_udpipe['processed_text'].apply(lambda x: x.split())
df_udpipe.to_csv("processed_texts_udpipe.csv", index=False, encoding='utf-8')
print("Результат предобработки UDPipe сохранен в processed_texts_udpipe.csv")

# ----------------------------


In [ ]:
def get_avg_pretrained_w2v(tokens, model):
    vectors = [model[token] for token in tokens if token in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df_udpipe['w2v_pretrained_features'] = df_udpipe['tokens'].apply(lambda tokens: get_avg_pretrained_w2v(tokens, pretrained_w2v))
w2v_pretrained_features_udpipe = np.vstack(df_udpipe['w2v_pretrained_features'].values)
w2v_pretrained_features_udpipe_df = pd.DataFrame(w2v_pretrained_features_udpipe,
                                                 columns=[f'w2v_ud_{i}' for i in range(w2v_pretrained_features_udpipe.shape[1])])
df_udpipe = pd.concat([df_udpipe.reset_index(drop=True), w2v_pretrained_features_udpipe_df.reset_index(drop=True)], axis=1)
df_udpipe.to_csv("udpipe_w2v_features.csv", index=False, encoding='utf-8')

# ----------------------------


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_rf = df_udpipe[[f'w2v_ud_{i}' for i in range(w2v_pretrained_features_udpipe.shape[1])]]
y_rf = df_udpipe['type']

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_rf, y_train_rf)
y_pred_rf = rf_model.predict(X_test_rf)

print("RandomForestClassifier (UDPipe Word2Vec features):")
print(classification_report(y_test_rf, y_pred_rf))
print(f"Accuracy: {accuracy_score(y_test_rf, y_pred_rf):.4f}")

# ----------------------------


In [ ]:
fasttext_model = FastText(sentences=df_udpipe['tokens'].tolist(), vector_size=100, window=5, min_count=1, workers=4)

def get_fasttext_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df_udpipe['fasttext_vector'] = df_udpipe['tokens'].apply(lambda tokens: get_fasttext_vector(tokens, fasttext_model))
fasttext_features = np.vstack(df_udpipe['fasttext_vector'].values)
fasttext_features_df = pd.DataFrame(fasttext_features, columns=[f'ft_{i}' for i in range(fasttext_features.shape[1])])
df_udpipe = pd.concat([df_udpipe.reset_index(drop=True), fasttext_features_df.reset_index(drop=True)], axis=1)
df_udpipe.to_csv("udpipe_fasttext_features.csv", index=False, encoding='utf-8')


In [ ]:
X_ft = df_udpipe[[f'ft_{i}' for i in range(fasttext_features.shape[1])]]
y_ft = df_udpipe['type']

X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(X_ft, y_ft, test_size=0.2, random_state=42)
rf_ft_model = RandomForestClassifier(random_state=42)
rf_ft_model.fit(X_train_ft, y_train_ft)
y_pred_ft = rf_ft_model.predict(X_test_ft)

print("RandomForestClassifier (FastText features):")
print(classification_report(y_test_ft, y_pred_ft))
print(f"Accuracy: {accuracy_score(y_test_ft, y_pred_ft):.4f}")

In [ ]:
pretrained_fasttext = KeyedVectors.load_word2vec_format('model.model', binary=True, encoding='utf-8', unicode_errors='ignore')

def get_avg_pretrained_fasttext(tokens, model):
    vectors = [model[word] for word in tokens if word in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df_udpipe['fasttext_pretrained_vector'] = df_udpipe['tokens'].apply(lambda tokens: get_avg_pretrained_fasttext(tokens, pretrained_fasttext))
fasttext_pretrained_features = np.vstack(df_udpipe['fasttext_pretrained_vector'].values)
fasttext_pretrained_df = pd.DataFrame(fasttext_pretrained_features, columns=[f'ft_pre_{i}' for i in range(fasttext_pretrained_features.shape[1])])
df_udpipe = pd.concat([df_udpipe.reset_index(drop=True), fasttext_pretrained_df.reset_index(drop=True)], axis=1)
df_udpipe.to_csv("udpipe_fasttext_pretrained_features.csv", index=False, encoding='utf-8')

# Финальная классификация: Logistic Regression на предобученных FastText-признаках
features_columns = [f'ft_pre_{i}' for i in range(fasttext_pretrained_features.shape[1])]
X_final = df_udpipe[features_columns]
y_final = df_udpipe['type']

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_final, y_final, test_size=0.2, random_state=42)
lr_final_model = LogisticRegression(max_iter=1000, random_state=42)
lr_final_model.fit(X_train_final, y_train_final)
y_pred_final = lr_final_model.predict(X_test_final)

print("Final Classification with Logistic Regression (Pretrained FastText features):")
print(classification_report(y_test_final, y_pred_final))
print(f"Precision: {precision_score(y_test_final, y_pred_final):.4f}")
print(f"Recall: {recall_score(y_test_final, y_pred_final):.4f}")
print(f"F1-score: {f1_score(y_test_final, y_pred_final):.4f}")

#Новая попытка


In [ ]:
from gensim.models import Word2Vec

# Обучение модели Word2Vec на токенах (столбец 'tokens' из df)
w2v_model = Word2Vec(sentences=df['tokens'].tolist(), vector_size=300, window=5, min_count=1, workers=4)

# Вывод похожих слов для 'хорошо'
print("Word2Vec – Слова, похожие на 'хорошо':")
print(w2v_model.wv.most_similar('хорошо'))

# Вывод примера словарного запаса (первые 10 слов)
print("\nWord2Vec – Пример словарного запаса (первые 10):")
print(list(w2v_model.wv.index_to_key)[:10])

# Вычисление усреднённого вектора для каждого документа (усреднение векторов слов)
w2v_vectors = []
for tokens in df['tokens']:
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if vectors:
        doc_vector = np.mean(vectors, axis=0)
    else:
        doc_vector = np.zeros(300)
    w2v_vectors.append(doc_vector)

# Создание DataFrame с меткой 'typr' и векторами
w2v_df = pd.concat([df[['typr']].reset_index(drop=True), pd.DataFrame(w2v_vectors)], axis=1)
w2v_df.to_csv('w2v.csv', index=False)

# Загрузка и вывод содержимого файла через DataFrame
df_w2v = pd.read_csv('w2v.csv')


In [ ]:
df_w2v

,typr,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,-0.304259,0.434958,-0.008827,-0.040342,0.115284,-0.264273,0.178683,0.795626,0.048046,...,-0.045260,0.424607,0.265564,0.120420,0.092982,0.370826,-0.164560,-0.389158,0.434415,-0.083041
1,-1,-0.156762,0.214352,0.303718,0.278041,0.295972,-0.436245,0.383107,0.958383,0.251319,...,-0.300008,0.505722,0.380714,0.214061,0.607170,0.504463,0.068286,-0.349237,0.722045,-0.071418
2,-1,-0.294935,0.942086,-0.281872,-0.047196,-0.153984,-0.536087,0.323158,1.109137,0.300751,...,0.449440,0.565022,0.493007,0.328588,0.280591,0.676266,0.038204,-0.415708,0.541390,-0.261491
3,-1,-0.018398,0.805867,0.039439,0.341452,-0.007830,-0.800662,0.395100,1.244850,0.339613,...,0.311432,0.627544,0.544129,0.431068,0.401445,0.484593,-0.094947,-0.160877,0.360131,-0.267431
4,-1,-0.223767,0.404206,0.229920,0.270405,-0.046320,-0.229201,0.338961,0.787914,-0.029858,...,-0.075361,0.366201,0.405985,0.139884,0.295282,0.496725,-0.059940,-0.320221,0.604854,-0.152911
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,1,-0.243709,0.103468,0.061630,0.259924,0.410867,-0.381347,0.562128,1.424526,0.225613,...,-0.127872,0.460221,0.270482,-0.033450,0.763934,0.495628,0.040171,-0.410666,0.644142,-0.154075
226830,1,-0.246534,0.382833,0.078876,0.277300,0.144939,-0.474135,0.349386,1.111710,0.047018,...,0.029015,0.361532,0.317706,0.162282,0.536084,0.411429,0.031325,-0.389500,0.580249,-0.215064
226831,1,0.125073,0.601882,0.013695,0.315129,0.064889,-0.616063,0.243789,1.193764,0.306941,...,0.033809,0.484172,0.612729,0.335495,0.642355,0.569276,0.017885,-0.324985,0.728014,-0.267063
226832,1,0.113162,0.682789,0.019004,0.306752,-0.283615,-0.496235,0.254145,0.956201,0.044405,...,0.139339,0.403712,0.621357,0.166096,0.292445,0.593253,0.006846,-0.221168,0.563390,-0.386809


In [ ]:
from gensim.models import KeyedVectors

try:
    # Загрузка предобученной модели из локального файла 'model.bin'
    pretrained_w2v = KeyedVectors.load_word2vec_format('model.bin', encoding='utf-8', unicode_errors='ignore', binary=True)
except Exception as e:
    print("Ошибка загрузки pretrained Word2Vec модели:", e)
    pretrained_w2v = None

if pretrained_w2v:
    # "Очистка" ключей: оставляем только слово до символа '_'
    w2v_cleaned = {key.split('_')[0]: pretrained_w2v[key] for key in pretrained_w2v.key_to_index}

    # Вычисление векторов для каждого документа (усреднение)
    w2v_pretrained_vectors = []
    for tokens in df['tokens']:
        vectors = [w2v_cleaned[word] for word in tokens if word in w2v_cleaned]
        if vectors:
            doc_vector = np.mean(vectors, axis=0)
        else:
            doc_vector = np.zeros(300)
        w2v_pretrained_vectors.append(doc_vector)

    w2v_pretrained_df = pd.concat([df[['typr']].reset_index(drop=True), pd.DataFrame(w2v_pretrained_vectors)], axis=1)
    w2v_pretrained_df.to_csv('w2v_pretrained.csv', index=False)


In [ ]:
w2v_pretrained_df

,typr,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,-0.325638,-0.363030,0.413511,0.179399,0.230704,-0.395714,0.394846,-0.252966,0.257283,...,-0.624421,-0.284439,-0.252557,0.516432,0.247912,0.548789,0.089924,-0.233948,0.100424,0.464523
1,-1,-0.081442,0.113996,0.161917,-0.185412,0.152473,0.064786,0.034131,-0.153709,-0.041807,...,0.014643,0.019055,0.173169,0.214876,-0.004609,-0.303236,0.148345,-0.109285,-0.232302,0.063928
2,-1,0.408046,-0.066171,0.681850,-0.177851,0.836231,0.021685,0.726518,-0.649751,0.000710,...,0.049154,-0.298492,-0.088222,0.644473,0.124620,-0.027587,0.143831,-0.300868,-0.338422,0.075640
3,-1,0.057314,-0.632438,0.521143,0.576838,-0.070294,-0.323442,0.263405,0.022823,-0.038259,...,0.051118,0.078557,-0.180014,0.885257,-0.297371,0.031147,-0.163688,0.090323,-0.273971,-0.159418
4,-1,-0.634761,-0.532511,-0.390650,0.299181,-0.013843,-0.490745,-0.192335,0.498148,0.061118,...,-0.463738,0.082606,-0.187021,-0.017703,0.239189,0.662629,-0.337891,0.403267,-0.211242,-1.013734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,1,-0.825148,-0.744613,-0.097037,-0.130733,0.342291,-0.312562,-0.105434,0.087703,0.185391,...,0.056417,0.213189,0.015083,0.104531,0.737710,0.004642,0.449725,0.146682,-1.306366,0.327593
226830,1,-0.055247,0.512450,0.389333,0.135722,0.221833,0.195144,0.477056,-0.379570,-0.454096,...,0.064714,0.031324,-0.184993,-0.806099,-0.142940,0.351284,-0.166041,0.369993,-0.185071,0.664518
226831,1,0.637349,-0.791408,0.590899,0.196915,0.144282,-0.556750,0.659850,0.167698,1.330819,...,-0.230145,0.133529,-0.669698,0.561082,0.062336,0.171485,0.006728,-0.751613,0.143604,-0.085217
226832,1,0.085820,-0.075950,-0.160474,0.225017,-0.076500,-0.388996,-0.386858,0.779886,0.496716,...,-0.612685,0.320777,0.039070,0.173838,-0.330444,-0.040214,0.249406,-0.650184,-0.514628,-0.015754


In [ ]:
from gensim.models import FastText

# Обучение модели FastText на токенах (используем min_count=5)
ft_model = FastText(sentences=df['tokens'].tolist(), vector_size=300, window=5, min_count=5, workers=4)

# Вывод похожих слов для 'хорошо'
try:
    print("\nFastText – Слова, похожие на 'хорошо':")
    print(ft_model.wv.most_similar('хорошо'))
except Exception as e:
    print("Ошибка при поиске похожих слов в FastText модели:", e)

# Вычисление векторного представления для каждого документа (усреднение)
ft_vectors = []
for tokens in df['tokens']:
    vectors = [ft_model.wv[word] for word in tokens if word in ft_model.wv]
    if vectors:
        doc_vector = np.mean(vectors, axis=0)
    else:
        doc_vector = np.zeros(300)
    ft_vectors.append(doc_vector)

ft_df = pd.concat([df[['typr']].reset_index(drop=True), pd.DataFrame(ft_vectors)], axis=1)
ft_df.to_csv('fasttext.csv', index=False)


In [ ]:
ft_df

,typr,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,0.147783,0.038190,-0.063589,-0.330981,0.253818,-0.344499,0.137013,0.126958,0.138864,...,0.007171,-0.271275,-0.824946,-0.039100,-0.284279,0.002936,0.698696,0.354474,0.750180,0.213504
1,-1,-0.033456,-0.193370,0.006665,-0.187794,0.760118,-0.230957,-0.348772,-0.140677,0.169002,...,-0.096766,-0.156719,-0.461544,-0.019235,0.146743,-0.342008,0.214196,0.072691,0.634970,0.578729
2,-1,0.032236,0.291269,-0.070991,-0.710657,-0.031977,-0.134281,0.118300,0.175111,0.169832,...,0.087802,-0.163184,-0.556536,0.133991,-0.526252,0.184330,0.097295,0.258740,0.507612,-0.196357
3,-1,-0.144791,0.226504,0.020300,-0.377292,0.409452,-0.470262,0.006073,0.339625,0.114362,...,0.208296,-0.029596,-0.810358,0.258107,-0.310297,-0.253220,0.387156,0.380015,0.667467,0.170164
4,-1,0.132129,0.002871,0.190067,-0.236085,0.710463,-0.268453,0.156919,-0.156528,0.288272,...,-0.092714,-0.165095,-0.424567,0.038166,-0.088761,-0.516653,0.101475,0.138995,0.696663,0.673323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,1,-0.053826,-0.060715,-0.144106,-0.310663,0.486594,-0.464382,-0.143001,-0.177522,-0.074581,...,0.001366,-0.341687,-1.132594,-0.143216,-0.299171,-0.100344,0.534073,-0.068341,1.070375,0.366554
226830,1,-0.007193,0.271357,0.012336,-0.465519,0.493177,-0.249943,0.036323,0.044502,-0.004121,...,-0.109786,-0.217799,-0.599781,-0.040117,-0.161937,-0.224058,0.295833,-0.073049,0.554033,0.143840
226831,1,-0.233005,0.127335,-0.174586,-0.140916,0.287103,-0.108269,-0.257747,0.130244,0.021090,...,0.128518,-0.100357,-0.487261,0.144341,-0.080201,-0.263255,0.035229,-0.087331,0.508601,0.035799
226832,1,0.018679,0.177153,-0.146248,-0.070144,-0.011037,-0.035943,0.259912,-0.042860,-0.244372,...,-0.226123,-0.084437,-0.383007,0.020087,-0.248209,-0.132330,0.150457,-0.039493,0.555234,0.135624


In [ ]:
try:
    # Загрузка предобученной модели FastText из файла 'ft_model.model'
    ft_pretrained_model = KeyedVectors.load('model.model')
except Exception as e:
    print("Ошибка загрузки pretrained FastText модели:", e)
    ft_pretrained_model = None

if ft_pretrained_model:
    ft_pretrained_vectors = []
    for tokens in df['tokens']:
        vectors = []
        for word in tokens:
            try:
                vectors.append(ft_pretrained_model[word])
            except KeyError:
                vectors.append(np.zeros(300))
        if vectors:
            doc_vector = np.mean(vectors, axis=0)
        else:
            doc_vector = np.zeros(300)
        ft_pretrained_vectors.append(doc_vector)

    ft_pretrained_df = pd.concat([df[['typr']].reset_index(drop=True), pd.DataFrame(ft_pretrained_vectors)], axis=1)
    ft_pretrained_df.to_csv('fasttext_pretr.csv', index=False)


In [ ]:
ft_pretrained_model.most_similar('кот')

In [ ]:
ft_pretrained_df

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# Словари для хранения accuracy по наборам признаков
results_lr = {}
results_dt = {}

# Функция для загрузки признаков и меток из файла.
# Если в файле отсутствует столбец с меткой, он добавляется из исходного DataFrame (df).
def get_vectors(path, label_col='typr'):
    data = pd.read_csv(path)
    if label_col not in data.columns:
        data[label_col] = df[label_col].values
    features = data.drop(label_col, axis=1)
    labels = data[label_col]
    return features, labels

# Функция обучения и оценки модели
def train_model(features, labels, model, data_type, model_name):
    X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\nОтчёт классификации для {model_name} на наборе '{data_type}':")
    print(classification_report(y_test, y_pred))
    return acc

# Инициализация моделей
lr_model = LogisticRegression(max_iter=1000)
dt_model = DecisionTreeClassifier(random_state=42)

# Словарь с наборами признаков и путями к файлам
feature_sets = {
    "linguistic_features": "linguistic_features.csv",
    "morphological_features": "morphological_features.csv",
    "tfidf_features": "tfidf_features.csv",   # Файл без метки, добавим метку из df
    "w2v": "w2v.csv",
    "w2v_pretrained": "w2v_pretrained.csv",
    "fasttext": "fasttext.csv",
    "fasttext_pretrained": "fasttext_pretr.csv"
}

# Для tfidf и морфологических признаков, если в файле нет метки, добавляем её из df
for name, path in feature_sets.items():
    features, labels = get_vectors(path)
    print(f"\n--- Обучение на наборе признаков: {name} ---")
    acc_lr = train_model(features, labels, lr_model, name, "Logistic Regression")
    results_lr[name] = acc_lr
    acc_dt = train_model(features, labels, dt_model, name, "Decision Tree")
    results_dt[name] = acc_dt

results_lr_df = pd.DataFrame.from_dict(results_lr, orient='index', columns=['Accuracy'])
results_dt_df = pd.DataFrame.from_dict(results_dt, orient='index', columns=['Accuracy'])

print("\nТочность Logistic Regression по наборам признаков:")
print(results_lr_df)
print("\nТочность Decision Tree по наборам признаков:")
print(results_dt_df)